# Conjugation boxes: Pauli gadgets and two-term LCU

**Download Notebook** - {nb-download}`conjugation_box.ipynb`

Build an XYZ Pauli gadget and a two-term LCU, then apply call, dagger, control,
and controlled dagger through `ConjugationBox`.

Run cells in order. Install dependencies with `uv sync --extra dev-dependencies`.

In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from scipy.linalg import expm

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, control, dagger, output
from guppylang.std.quantum import (
    collect_measurements, cx, cz, h, measure_array, qubit, ry, rz, sdg, x,
)

from guppyalgos.primitives.subroutines.conjugation import ConjugationBox
from guppyalgos.testing import assert_unitary_modifiers
from guppyalgos.utils import qarray

# Guppy angles are in half-turns: the angle in radians is alpha = pi * theta.
theta = 0.37
alpha = np.pi * theta

## 1. What does the box do?

G must be daggerable and U unitary. Both act on the same register.
`box.compose(qs)` executes **G → U → G†**; matrices act right to left:

$$
W=G^{\dagger}UG,
\qquad
W^{\dagger}=G^{\dagger}U^{\dagger}G.
$$

| Call context | Execution order |
| --- | --- |
| Ordinary call | G → U → G† |
| `with dagger:` | G → U† → G† |
| `with control(controls):` | G → controlled-U → G† |
| `with control(controls), dagger:` | G → controlled-U† → G† |

Only U receives the external control c:

$$
\mathrm{C}(W)=(I_c\otimes G^{\dagger})\,\mathrm{C}(U)\,(I_c\otimes G).
$$

- c = 0: U is skipped and G† G cancels.
- c = 1: the box applies W.

Use U† in place of U for the controlled adjoint.

## 2. An XYZ Pauli gadget

For $P=X_0Y_1Z_2$, G applies H on q0, S† then H on q1, and two parity CNOTs.
This gives $G^{\dagger}Z_2G=P$. With $U=R_z(\alpha)$ on q2 and
$\alpha=\pi\theta$:

$$
W_{XYZ}(\theta)=G^{\dagger}R_z(\alpha)G=e^{-i\alpha X_0Y_1Z_2/2}.
$$

In [2]:
@guppy(daggerable=True)
def g(qs: array[qubit, 3]) -> None:
    h(qs[0])
    sdg(qs[1])
    h(qs[1])
    cx(qs[0], qs[2])
    cx(qs[1], qs[2])


@guppy(unitary=True)
def u(qs: array[qubit, 3]) -> None:
    rz(qs[2], angle(theta))

### Call, dagger, control, and controlled dagger

The program below exercises all four modes with a superposed control.
Each call/inverse pair cancels, so undoing the initial H gates leaves all zeros.

Only the middle $\mathrm{C}R_z(\alpha)$ receives the external control;
the basis changes and parity CNOTs remain uncontrolled.

In [3]:
@guppy
def run_pauli_gadget() -> None:
    qs = qarray(3)
    controls = qarray(1)
    h(qs[0])
    h(controls[0])
    box = ConjugationBox(g, u)

    box.compose(qs)
    with dagger:
        box.compose(qs)
    with control(controls):
        box.compose(qs)
    with control(controls), dagger:
        box.compose(qs)

    h(qs[0])
    h(controls[0])
    output("targets", collect_measurements(measure_array(qs)))
    output("control", collect_measurements(measure_array(controls)))


pauli_counts = (
    run_pauli_gadget.emulator(4)
    .with_seed(42)
    .with_shots(8)
    .run()
    .collated_counts()
)
print(pauli_counts)

Counter({(('targets', '000'), ('control', '0')): 8})


### Verify the complete operation

Construct $e^{-i\alpha P/2}$ with Zixy in big-endian order.
`assert_unitary_modifiers` checks all four modes, including the inactive control
branch.

In [4]:
@guppy(unitary=True)
def pauli_gadget(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    box.compose(qs)


pauli_hamiltonian = zqp.RealTermSum.from_str("(1.0, X0 Y1 Z2)", 3)
pauli_matrix = pauli_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
pauli_reference = expm(-0.5j * alpha * pauli_matrix)

assert_unitary_modifiers(pauli_gadget, 3, expected_unitary=pauli_reference)
print("XYZ gadget: all four modifier modes match the reference matrices.")

XYZ gadget: all four modifier modes match the reference matrices.


## 3. A two-term LCU block encoding

Use `qs[0]` as index a and `qs[1]` as target t. G = PREPARE rotates a:

$$
R_y(\alpha)|0\rangle=c_0|0\rangle+c_1|1\rangle,
\qquad
c_0=\cos(\alpha/2),\quad c_1=\sin(\alpha/2).
$$

U = SELECT applies X for index zero and Z for index one:

$$
\mathrm{SELECT}=|0\rangle\langle0|_a\otimes X_t
+|1\rangle\langle1|_a\otimes Z_t.
$$

The box implements

$$
B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE},
$$

with index-zero block

$$
(\langle0|_a\otimes I_t)\,B\,(|0\rangle_a\otimes I_t)
=c_0^2X_t+c_1^2Z_t=A.
$$

B is unitary; A need not be. Keep the index qubit when checking B's modifiers.

In [5]:
@guppy(daggerable=True)
def prepare(qs: array[qubit, 2]) -> None:
    ry(qs[0], angle(theta))


@guppy(unitary=True)
def select(qs: array[qubit, 2]) -> None:
    # Open-index control: apply X to the target when the index is zero.
    x(qs[0])
    cx(qs[0], qs[1])
    x(qs[0])
    # Closed-index control: apply Z to the target when the index is one.
    cz(qs[0], qs[1])

### Distinguish the index from the external control

Index a chooses X or Z. External control c turns the whole SELECT block on or off.

- c = 0: SELECT is skipped and PREPARE† PREPARE cancels.
- c = 1: SELECT uses a to choose the Pauli.
- PREPARE and PREPARE† always run.

The control covers SELECT's open-index X gates too. It controls the full block
encoding B, including its index qubit, rather than A alone.

In [6]:
@guppy
def run_lcu() -> None:
    qs = qarray(2)
    controls = qarray(1)
    h(qs[1])
    h(controls[0])
    box = ConjugationBox(prepare, select)

    box.compose(qs)
    with dagger:
        box.compose(qs)
    with control(controls):
        box.compose(qs)
    with control(controls), dagger:
        box.compose(qs)

    h(qs[1])
    h(controls[0])
    output("index_and_target", collect_measurements(measure_array(qs)))
    output("control", collect_measurements(measure_array(controls)))


lcu_counts = (
    run_lcu.emulator(3)
    .with_seed(42)
    .with_shots(8)
    .run()
    .collated_counts()
)
print(lcu_counts)

Counter({(('index_and_target', '00'), ('control', '0')): 8})


### Verify the unitary and its LCU block

Build the reference with Zixy using

$$
\mathrm{SELECT}=\tfrac12(X_t+Z_aX_t+Z_t-Z_aZ_t).
$$

Check all four modifier modes of B and its top-left block $c_0^2X+c_1^2Z$.

In [7]:
@guppy(unitary=True)
def lcu(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    box.compose(qs)


prepare_hamiltonian = zqp.RealTermSum.from_str("(1.0, Y0)", 2)
prepare_matrix = expm(
    -0.5j * alpha * prepare_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
)
select_hamiltonian = zqp.RealTermSum.from_str(
    "(0.5, X1), (0.5, Z0 X1), (0.5, Z1), (-0.5, Z0 Z1)", 2
)
select_matrix = select_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
lcu_reference = prepare_matrix.conj().T @ select_matrix @ prepare_matrix

assert_unitary_modifiers(lcu, 2, expected_unitary=lcu_reference)

weight_x = np.cos(alpha / 2) ** 2
weight_z = np.sin(alpha / 2) ** 2
lcu_hamiltonian = zqp.RealTermSum.from_str(
    f"({weight_x}, X0), ({weight_z}, Z0)", 1
)
expected_block = lcu_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
np.testing.assert_allclose(lcu_reference[:2, :2], expected_block, atol=1e-10)

print(f"LCU block: {weight_x:.6f} X + {weight_z:.6f} Z")
print("LCU: all four modifier modes and the projected block match.")

LCU block: 0.698574 X + 0.301426 Z
LCU: all four modifier modes and the projected block match.
